# Finance AI Agent using OpenAI Agents SDK + Ollama

## Objective

In this notebook, we build a set of **Finance AI Agents** that can answer stock
market questions and produce a **blunt, honest sentiment forecast** for a stock,
using:

- **OpenAI Agents SDK** – gives our AI a way to call Python functions ("tools")
- **Ollama** – runs a large language model (LLM) locally on our own machine
- **yfinance** – live stock price, analyst recommendations, and quarterly financials
- **ddgs** – general web search (public news, opinion, discussion)
- **gnews** – Google News headlines
- **edgartools** – official SEC 10-Q filing text
- **Gradio** – gives our agents a simple web page (UI) to chat with

## What the Agents Can Do

1. **Finance Agent** — current stock price + analyst recommendations
2. **News Sentiment Agent** — reads Yahoo Finance news and judges tone
3. **Sentiment Forecast Agent** — the main new addition. It gathers information
   from **5 different tools** (Yahoo news, web search, Google News, quarterly
   financials, and SEC 10-Q filings) and writes one detailed report that is
   **blunt and direct** about:
   - Positive / Negative / Neutral tone
   - Fear or uncertainty in the market
   - Anger / public distrust toward the company
   - Management or board-level disputes
   - Business strategy shifts (checked against the actual quarterly numbers)
   - An overall outlook heading into the **next trading day**

> **Important honesty note:** this agent forecasts *sentiment*, not a guaranteed
> stock price. Real markets are affected by many more factors than sentiment
> alone. The agent will say this plainly in its own report too.

## Before You Start

Install the extra libraries used in this notebook (one time, in your terminal):

```
pip install ddgs gnews edgartools
```

`edgartools` also requires you to identify yourself to SEC EDGAR (name + email)
— this is free, no API key, just good manners so SEC knows who is requesting
data. We set this in Step 2 below.

---

## Step 1: Connect to our Local Ollama Model

The OpenAI Agents SDK normally talks to OpenAI's cloud servers. Since we want to
run everything **locally and for free** using **Ollama**, we need to tell the SDK
two things:

- **Where to send requests** → our local Ollama server (`http://localhost:11434/v1`)
- **What API key to use** → Ollama doesn't need a real key, so we just use the
  placeholder text `"ollama"`

> **Note:** Make sure Ollama is installed and running on your computer before
> running this notebook. You can check by opening `http://localhost:11434` in
> your browser.

In [13]:
# We use the "os" library to set environment variables.
# Environment variables are a way to store settings that our program will read.
import os

# Tell the OpenAI Agents SDK to send requests to our local Ollama server
# instead of the real OpenAI cloud servers.
os.environ["OPENAI_BASE_URL"] = "http://localhost:11434/v1"

# Ollama does not check for a real API key, so we just use a placeholder value.
os.environ["OPENAI_API_KEY"] = "ollama"


## Step 2: Import the Libraries We Need

Here we import every library required for this project:

- `yfinance` → stock price, analyst recommendations, quarterly financials
- `agents` (OpenAI Agents SDK) → to create AI agents and give them tools
- `asyncio` → to run multiple agents at the same time (in parallel)
- `gradio` → to build a simple web page for chatting with our agents
- `ddgs` → general web search (public news, discussion, opinion)
- `gnews` → Google News headlines
- `edgar` (from `edgartools`) → official SEC filings (10-Q quarterly reports)

We also set our **identity** for SEC EDGAR here. SEC asks that every requester
identify themselves with a name and email — replace the placeholder below with
your own details.

In [14]:
# Library to fetch stock market data from Yahoo Finance
import yfinance as yf

# Core building blocks from the OpenAI Agents SDK
from agents import (
    Agent,              # Used to create an AI agent
    Runner,             # Used to run an agent and get its response
    function_tool,      # Decorator that turns a normal Python function into a tool
    set_tracing_disabled # Used to turn off cloud tracing/logging
)

# Library that lets us run multiple tasks (like our agents) at the same time
import asyncio

# Library to quickly build a simple web-based chat interface
import gradio as gr

# General web search library (no API key needed)
from ddgs import DDGS

# Google News headlines library (no API key needed)
from gnews import GNews

# SEC EDGAR library - lets us download official company filings (like 10-Q reports)
from edgar import Company, set_identity

# SEC requires every requester to identify themselves with a name and email.
# Replace this with your own name and email before running.
set_identity("Your Name your.email@example.com")


## Step 3: Disable Cloud Tracing

The OpenAI Agents SDK can send "traces" (logs of what the agent is doing) to
OpenAI's cloud dashboard. This is useful when using the real OpenAI API, but
since we are running everything **locally with Ollama**, this cloud tracing
is unnecessary. We disable it below.

In [15]:
# Turn off tracing so the SDK does not try to send logs to OpenAI's cloud
set_tracing_disabled(True)


## Step 4: Tool 1 — Get Current Stock Price

Our AI agent cannot fetch live stock prices on its own — LLMs only know text
patterns, not real-time data. So we give it a **tool**: a normal Python
function that the agent can call whenever it needs a stock price.

**Input:** A ticker symbol (for example: `TSLA`, `AAPL`, `MSFT`)

**Output:** A text message containing the latest stock price

The `@function_tool` decorator (from the Agents SDK) is what makes this
regular Python function usable by our AI agent.

In [16]:
# ============================================================
# Tool: Get Current Stock Price
# ============================================================

@function_tool
def get_stock_price(ticker: str) -> str:
    """
    Fetch the latest closing stock price for the given ticker symbol.
    """

    # Create a Yahoo Finance object for the given ticker (e.g. "TSLA")
    stock = yf.Ticker(ticker)

    # Download the last 1 day of price history for this stock
    history = stock.history(period="1d")

    # Get the closing price from the very last row of data
    latest_close = history["Close"].iloc[-1]

    # Return a simple, readable text message with the price
    return f"The current price of {ticker} is ${latest_close:.2f}"


## Step 5: Tool 2 — Analyst Recommendations

This tool lets our agent fetch **analyst ratings** (Buy / Hold / Sell etc.)
for a stock from Yahoo Finance.

- If recommendations are available, we return the **10 most recent** ones.
- If no recommendations exist for that ticker, we return a friendly message
  instead of an error.

In [17]:
# ============================================================
# Tool: Get Analyst Recommendations
# ============================================================

@function_tool
def get_analyst_recommendations(ticker: str) -> str:
    """
    Retrieve analyst recommendations for the specified stock ticker.
    """

    # Create a Yahoo Finance object for the given ticker
    stock = yf.Ticker(ticker)

    # Download the analyst recommendation data (as a table/DataFrame)
    recommendations = stock.recommendations

    # If there is no data available, tell the user instead of returning an error
    if recommendations is None or recommendations.empty:
        return f"No analyst recommendations found for {ticker}"

    # Keep only the 10 most recent recommendations
    latest_recommendations = recommendations.tail(10)

    # Convert the table into a nicely formatted markdown table (text)
    return latest_recommendations.to_markdown()


## Step 6: Explore the News Data (Optional)

Before writing our news tool, it helps to look at the **raw data** that
`yfinance` gives us for stock news. This way we know exactly which fields
(like `title`, `summary`, `publisher`) we need to pull out.

This cell is just for exploration — it prints one sample news item for
Tesla (`TSLA`) so we can see its structure.

In [18]:
# Look at the raw structure of one news item for TSLA.
# This helps us understand what fields are available before building our tool.
stock = yf.Ticker("TSLA")
print(stock.news[:1])


DNSError: Failed to perform, curl: (6) Could not resolve host: query1.finance.yahoo.com. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

## Step 7: Tool 3 — Get Stock News (Yahoo Finance)

Using what we learned from exploring the data above, we can now build a tool
that fetches the **latest news headlines and summaries** for a given stock
directly from Yahoo Finance.

In [ ]:
# ============================================================
# Tool: Get Stock News (Yahoo Finance)
# ============================================================

@function_tool
def get_stock_news(ticker: str) -> str:
    """
    Fetch latest news headlines and summaries for the given stock ticker
    from Yahoo Finance.
    """

    # Create a Yahoo Finance object and fetch its news list
    stock = yf.Ticker(ticker)
    news_items = stock.news

    # If there is no news at all, return a friendly message
    if not news_items:
        return f"No recent news found for {ticker}"

    # This list will hold the formatted text for each news article
    articles = []

    # Only look at the first 10 news items
    for item in news_items[:10]:
        # Each news item stores its details inside a "content" dictionary
        content = item.get("content", {})

        # Pull out the fields we care about, with safe defaults if missing
        title = content.get("title", "No title")
        summary = content.get("summary", "")
        provider = content.get("provider", {})
        publisher = provider.get("displayName", "Unknown source")

        # Skip any article that has no real title (likely bad/incomplete data)
        if title == "No title":
            continue

        # Format the article as "- [Publisher] Title \n  Summary"
        articles.append(f"- [{publisher}] {title}\n  {summary}")

    # If nothing useful was collected, say so
    if not articles:
        return f"No recent news found for {ticker}"

    # Join all articles into one big text block, separated by new lines
    return "\n".join(articles)


## Step 8: Tool 4 — Web News Search (ddgs)

Yahoo Finance news alone is limited to Yahoo's own partner publishers. To
capture **broader public sentiment** (opinion pieces, discussion, smaller
outlets), we add a general **web search** tool using `ddgs` (the DuckDuckGo
search library). No API key is required.

In [ ]:
# ============================================================
# Tool: Get Web News (DuckDuckGo Search via ddgs)
# ============================================================

@function_tool
def get_web_news_ddgs(ticker: str) -> str:
    """
    Search the general web (via DuckDuckGo) for recent news and public
    discussion about the given stock ticker.
    """

    # Build a simple, specific search query
    query = f"{ticker} stock news"

    # Run the search and ask for up to 10 results
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=10))

    # If nothing came back, say so
    if not results:
        return f"No web search results found for {ticker}"

    # This list will hold the formatted text for each search result
    articles = []

    for result in results:
        title = result.get("title", "No title")
        body = result.get("body", "")
        link = result.get("href", "")

        articles.append(f"- {title}\n  {body}\n  Source: {link}")

    # Join all results into one big text block, separated by new lines
    return "\n".join(articles)


## Step 9: Tool 5 — Web News Search (gnews)

As a second, independent news source, we add `gnews`, a simple wrapper around
**Google News**. This often surfaces different articles than `ddgs`, giving us
a wider spread of public sentiment. No API key is required.

In [ ]:
# ============================================================
# Tool: Get Web News (Google News via gnews)
# ============================================================

@function_tool
def get_web_news_gnews(ticker: str) -> str:
    """
    Search Google News for recent headlines about the given stock ticker.
    """

    # Create a GNews object, limited to the 10 most recent results
    google_news = GNews(max_results=10)

    # Search for news about this ticker
    news_items = google_news.get_news(f"{ticker} stock")

    # If nothing came back, say so
    if not news_items:
        return f"No Google News results found for {ticker}"

    # This list will hold the formatted text for each news article
    articles = []

    for item in news_items:
        title = item.get("title", "No title")
        description = item.get("description", "")
        publisher_info = item.get("publisher", {})
        publisher = publisher_info.get("title", "Unknown source")

        articles.append(f"- [{publisher}] {title}\n  {description}")

    # Join all articles into one big text block, separated by new lines
    return "\n".join(articles)


## Step 10: Tool 6 — Quarterly Financial Report (yfinance)

Sentiment forecasting is more reliable when it is checked against real
numbers. This tool pulls the company's **quarterly financial statement**
(revenue, net income, and other key line items) from Yahoo Finance, so the
agent can tell whether a "strategy shift" mentioned in the news is actually
backed up by the financials.

In [ ]:
# ============================================================
# Tool: Get Quarterly Financial Report
# ============================================================

@function_tool
def get_quarterly_report(ticker: str) -> str:
    """
    Fetch the company's quarterly financial statement (revenue, net income,
    and other key line items) for the given stock ticker.
    """

    # Create a Yahoo Finance object for the given ticker
    stock = yf.Ticker(ticker)

    # Download the quarterly financial statement (as a table/DataFrame)
    quarterly_financials = stock.quarterly_financials

    # If there is no data available, tell the user instead of returning an error
    if quarterly_financials is None or quarterly_financials.empty:
        return f"No quarterly financial report found for {ticker}"

    # Convert the table into a nicely formatted markdown table (text)
    return quarterly_financials.to_markdown()


## Step 11: Tool 7 — SEC EDGAR Quarterly Filing (10-Q)

The most detailed and official source of information is the company's own
**10-Q filing**, submitted to the SEC every quarter. It contains management's
own commentary on strategy, risks, and performance — in their own words.

Because a full 10-Q filing can be very long, we only keep the **first part**
of the document text (enough for the agent to pick up key commentary, without
overwhelming it with hundreds of pages).

In [ ]:
# ============================================================
# Tool: Get SEC EDGAR 10-Q Filing Text
# ============================================================

@function_tool
def get_edgar_filing(ticker: str) -> str:
    """
    Fetch the text of the most recent 10-Q (quarterly report) filing for the
    given stock ticker directly from SEC EDGAR.
    """

    try:
        # Look up the company on SEC EDGAR using its ticker symbol
        company = Company(ticker)

        # Get the most recent 10-Q filing
        filing = company.get_filings(form="10-Q").latest()

        # Extract the plain text of the filing
        filing_text = filing.text()

        # 10-Q filings can be very long (hundreds of pages), so we only keep
        # the first 3000 characters - enough to capture key commentary
        trimmed_text = filing_text[:3000]

        return trimmed_text

    except Exception as error:
        # If anything goes wrong (ticker not found, no filings, etc.),
        # return a friendly message instead of crashing
        return f"Could not retrieve SEC filing for {ticker}. Reason: {error}"


## Step 12: Create the AI Agents

Now that all our tools are ready, we create **three** AI agents:

1. **Finance Agent** — answers questions about stock **price** and
   **analyst recommendations**.
2. **News Sentiment Agent** — reads Yahoo Finance news and gives a quick
   Positive / Negative / Mixed read.
3. **Sentiment Forecast Agent** — the main agent for this notebook. It calls
   **all 5 information tools** (Yahoo news, DuckDuckGo web search, Google
   News, quarterly financials, and SEC 10-Q filing) and produces one detailed,
   **blunt and honest** report.

All three agents run on the same local model (`gemma4:31b-cloud`) served
through Ollama, but each is given different tools and instructions.

In [ ]:
# ============================================================
# Create the Finance AI Agents
# ============================================================

# This agent focuses on stock price and analyst recommendations
finance_agent = Agent(
    name="Finance Agent",
    instructions=(
        "You are a helpful finance assistant. "
        "Whenever appropriate, present structured information using tables."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_price, get_analyst_recommendations],
)

# This agent focuses on reading Yahoo Finance news and judging sentiment
news_sentiment_agent = Agent(
    name="News Sentiment Agent",
    instructions=(
        "You ALWAYS analyze stock news sentiment immediately - never ask for "
        "permission or confirmation. "
        "When given any query mentioning a company or ticker, immediately call "
        "get_stock_news for that ticker. "
        "Do not ask the user if they want news analysis - just do it. "
        "Steps: "
        "1. Call get_stock_news with the ticker extracted from the query. "
        "2. Judge overall tone: Positive, Negative, or Mixed/Neutral. "
        "3. Give a short summary (3-5 bullet points) in simple, plain English. "
        "4. End with one line: 'Overall Sentiment: <Positive/Negative/Mixed>'. "
        "Never respond with a question. Always respond with the analysis."
    ),
    model="gemma4:31b-cloud",
    tools=[get_stock_news],
)

# This agent produces the main, detailed sentiment forecast report.
# It uses ALL 5 information-gathering tools together.
sentiment_forecast_agent = Agent(
    name="Sentiment Forecast Agent",
    instructions=(
        "You are a blunt, direct financial sentiment analyst. You never soften "
        "bad news and you never ask for permission before acting. "
        "As soon as you are given a query mentioning a company or ticker, "
        "immediately call ALL FIVE of your tools for that ticker: "
        "get_stock_news, get_web_news_ddgs, get_web_news_gnews, "
        "get_quarterly_report, and get_edgar_filing. "
        ""
        "Then write ONE detailed, well organized report with these sections: "
        ""
        "1. 'News Summary' - what Yahoo Finance and web news are reporting, "
        "in plain English. "
        "2. 'Public Sentiment' - what the general web/public discussion "
        "sounds like (from ddgs and gnews results). "
        "3. 'Financial Health Check' - what the quarterly report numbers show "
        "(revenue, net income trend - growing, shrinking, or flat). "
        "4. 'Filing Insights' - anything notable in the 10-Q filing text, "
        "especially strategy changes, risk warnings, or management commentary. "
        "Compare any claimed strategy shift against the actual quarterly "
        "numbers - point out clearly if they do NOT match. "
        "5. 'Sentiment Tags' - explicitly state which of the following apply, "
        "and why, using blunt and direct language, do not be diplomatic: "
        "Positive, Negative, Neutral, Fear, Anger/Distrust, "
        "Management Dispute, Strategy Shift Risk. More than one tag can apply "
        "at once. If there is fear, anger, or public distrust, say so plainly "
        "with the evidence you found - do not hide or soften it. If there is a "
        "leadership or board dispute, name it clearly. "
        "6. 'Outlook For Next Trading Day' - a short, honest paragraph "
        "describing the overall mood and expected sentiment direction heading "
        "into the next trading session, based only on what you found above. "
        ""
        "Always finish with this exact disclaimer on its own line: "
        "'Note: this is a sentiment read based on news, public discussion, and "
        "filings - not a guaranteed price prediction. Actual stock prices "
        "depend on many additional factors.'"
    ),
    model="gemma4:31b-cloud",
    tools=[
        get_stock_news,
        get_web_news_ddgs,
        get_web_news_gnews,
        get_quarterly_report,
        get_edgar_filing,
    ],
)


## Step 13: Run All Three Agents Together

We want all three agents to answer the same question at the same time,
instead of waiting for one to finish before starting the next. We do this
using `asyncio`, which lets Python run multiple tasks **in parallel**.

We write two small helper functions:

1. `run_all_agents` — an **async** function that starts all three agents at
   once and waits for all of them to finish.
2. `chat_with_agents` — a normal (non-async) function that Gradio can call.
   It simply runs `run_all_agents` behind the scenes.

> **Note:** The Sentiment Forecast Agent calls 5 tools (including a live web
> search and an SEC filing download), so it will naturally take longer to
> respond than the other two agents.

In [ ]:
# This function runs all three agents at the same time and waits for all results
async def run_all_agents(user_input: str):
    # Start the Finance Agent (this does not block/wait yet)
    price_task = Runner.run(finance_agent, user_input)

    # Start the News Sentiment Agent (this also does not block/wait yet)
    sentiment_task = Runner.run(news_sentiment_agent, user_input)

    # Start the Sentiment Forecast Agent (this also does not block/wait yet)
    forecast_task = Runner.run(sentiment_forecast_agent, user_input)

    # Now wait for all three agents to finish, running them in parallel
    price_result, sentiment_result, forecast_result = await asyncio.gather(
        price_task, sentiment_task, forecast_task
    )

    # Return the final text answer from each agent
    return (
        price_result.final_output,
        sentiment_result.final_output,
        forecast_result.final_output,
    )


In [ ]:
# Gradio does not work directly with "async" functions, so we create a
# normal function that runs our async function for us using asyncio.run().
'''
def chat_with_agents(user_input: str):
    price_output, sentiment_output, forecast_output = asyncio.run(
        run_all_agents(user_input)
    )
    return price_output, sentiment_output, forecast_output
'''
'''
def chat_with_agents(user_input: str):
    # First, immediately show a loading message in all three boxes
    yield "⏳ Fetching stock price...", "⏳ Analyzing news sentiment...", "⏳ Gathering full sentiment forecast (this can take a bit longer)..."

    # Now actually run the agents and wait for the real results
    price_output, sentiment_output, forecast_output = asyncio.run(
        run_all_agents(user_input)
    )

    # Yield again with the final results - this replaces the loading text
    yield price_output, sentiment_output, forecast_output
'''

'''
def chat_with_agents(user_input: str):
    yield (
        "### 💰 Finance Agent\n---\n⏳ Fetching stock price...",
        "### 📰 News Sentiment Agent\n---\n⏳ Analyzing news sentiment...",
        "### 🔮 Sentiment Forecast Agent\n---\n⏳ Gathering full sentiment forecast (this can take a bit longer)..."
    )

    price_output, sentiment_output, forecast_output = asyncio.run(
        run_all_agents(user_input)
    )

    yield (
        f"### 💰 Finance Agent\n---\n{price_output}",
        f"### 📰 News Sentiment Agent\n---\n{sentiment_output}",
        f"### 🔮 Sentiment Forecast Agent\n---\n{forecast_output}"
    )

'''


async def chat_with_agents(user_input: str):
    price_text = "###  Finance Agent\n---\n⏳ Fetching stock price..."
    sentiment_text = "###  News Sentiment Agent\n---\n⏳ Analyzing news sentiment..."
    forecast_text = "###  Sentiment Forecast Agent\n---\n⏳ Gathering full sentiment forecast..."

    yield price_text, sentiment_text, forecast_text

    # Small helper: runs an agent and returns its name together with the result,
    # so we always know which agent a finished task belongs to.
    async def run_named(name, agent):
        try:
            result = await Runner.run(agent, user_input)
            return name, result.final_output
        except Exception as error:
            return name, f" This agent failed: {error}"

    # Create the three named tasks
    named_tasks = [
        run_named("price", finance_agent),
        run_named("sentiment", news_sentiment_agent),
        run_named("forecast", sentiment_forecast_agent),
    ]

    # As each one finishes (in whatever order), update just that box
    for finished in asyncio.as_completed(named_tasks):
        agent_name, output_text = await finished

        if agent_name == "price":
            price_text = f"###  Finance Agent\n---\n{output_text}"
        elif agent_name == "sentiment":
            sentiment_text = f"###  News Sentiment Agent\n---\n{output_text}"
        elif agent_name == "forecast":
            forecast_text = f"###  Sentiment Forecast Agent\n---\n{output_text}"

        yield price_text, sentiment_text, forecast_text

## Step 14: Build and Launch the Web Interface

Finally, we use **Gradio** to create a simple chat-style web page where a
user can type a question (for example: *"What is the current price of
TSLA?"*) and see three separate answers side by side:

1. **Stock Price / Recommendations** — from the Finance Agent
2. **News Sentiment (Quick Read)** — from the News Sentiment Agent
3. **Detailed Sentiment Forecast Report** — from the Sentiment Forecast Agent

Running the cell below will start a local web server and open the app in
your browser.

In [ ]:
# Build the web interface using Gradio
'''
demo = gr.Interface(
    fn=chat_with_agents,  # the function to call when the user submits a question
    inputs=gr.Textbox(
        label="Ask about a stock",
        placeholder="e.g. What is the sentiment forecast for TSLA?"
    ),
    outputs=[
    gr.Markdown(label="Stock Price / Recommendations"),
    gr.Markdown(label="News Sentiment (Quick Read)"),
    gr.Markdown(label="Detailed Sentiment Forecast Report"),
    ],
    title="Finance AI Agent",
    description="Get stock price info, a quick sentiment read, and a detailed, blunt sentiment forecast report - side by side.",
)

# Launch the web app (this will open a local URL you can visit in your browser)
demo.launch()
'''

with gr.Blocks() as demo:
    gr.Markdown("# Finance AI Agent")
    gr.Markdown("Get stock price info, a quick sentiment read, and a detailed, blunt sentiment forecast report - side by side.")

    user_input = gr.Textbox(label="Ask about a stock", placeholder="e.g. What is the sentiment forecast for TSLA?")

    with gr.Row():
        submit_btn = gr.Button("Submit", variant="primary")
        clear_btn = gr.Button("Clear")

    with gr.Row():
        with gr.Group():
            price_box = gr.Markdown()
        with gr.Group():
            sentiment_box = gr.Markdown()
        with gr.Group():
            forecast_box = gr.Markdown()

    submit_btn.click(chat_with_agents, inputs=user_input, outputs=[price_box, sentiment_box, forecast_box])
    clear_btn.click(lambda: ("", "", ""), outputs=[price_box, sentiment_box, forecast_box])

demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
